In [1]:
import os
import numpy as np
from torch_geometric.data import Data, InMemoryDataset
import torch
from torch_cluster import knn_graph
from torch.nn import Sequential, Linear, ReLU
import torch.nn.functional as F
from torch_geometric.nn import global_max_pool, global_mean_pool, MessagePassing
from torch_geometric.nn.inits import reset
from torch import Tensor

In [2]:
import pytorch3d
from pytorch3d.loss import chamfer_distance

In [3]:
folder_list = os.listdir('./sphere_data')

In [4]:
data = []
count2 = 0
for folder in folder_list:
    thickness = folder.split('_')[2]
    u = folder.split('_')[3]
    folder_path = os.path.join('./sphere_data',folder)
    gi_path = os.path.join('./sphere_data_gi',folder)
    gi_file_name = thickness+'_'+u+'.txt'
    gi_file_name_path = os.path.join(gi_path,gi_file_name)
    gi = np.loadtxt(gi_file_name_path)
    num = len(os.listdir(folder_path))
    for i in range(num):
        count = i*500
        file_name = thickness+'_'+u+'_'+str(count)+'.txt'
        file_name_path = os.path.join(folder_path,file_name)
        data1 = np.loadtxt(file_name_path)
        e_i = knn_graph(torch.from_numpy(data1[:,:3]), k=3, loop=True).to('cuda')
        data.append(Data(x=torch.from_numpy(data1[:,:3]).to('cuda',dtype=torch.float), edge_index = e_i, g = torch.from_numpy(np.array(gi[i])).to('cuda',dtype=torch.float), curv = torch.from_numpy(data1[:,6]).to('cuda',dtype=torch.float)))
        print(count2)
        print(file_name_path)
        count2+=1

0
./pt-brain-2/res_f_0.032_u085/0.032_u085_0.txt
1
./pt-brain-2/res_f_0.032_u085/0.032_u085_500.txt
2
./pt-brain-2/res_f_0.032_u085/0.032_u085_1000.txt
3
./pt-brain-2/res_f_0.032_u085/0.032_u085_1500.txt
4
./pt-brain-2/res_f_0.032_u085/0.032_u085_2000.txt
5
./pt-brain-2/res_f_0.032_u085/0.032_u085_2500.txt
6
./pt-brain-2/res_f_0.032_u085/0.032_u085_3000.txt
7
./pt-brain-2/res_f_0.032_u085/0.032_u085_3500.txt
8
./pt-brain-2/res_f_0.032_u085/0.032_u085_4000.txt
9
./pt-brain-2/res_f_0.032_u085/0.032_u085_4500.txt
10
./pt-brain-2/res_f_0.032_u085/0.032_u085_5000.txt
11
./pt-brain-2/res_f_0.032_u085/0.032_u085_5500.txt
12
./pt-brain-2/res_f_0.032_u085/0.032_u085_6000.txt
13
./pt-brain-2/res_f_0.032_u085/0.032_u085_6500.txt
14
./pt-brain-2/res_f_0.032_u085/0.032_u085_7000.txt
15
./pt-brain-2/res_f_0.032_u085/0.032_u085_7500.txt
16
./pt-brain-2/res_f_0.032_u085/0.032_u085_8000.txt
17
./pt-brain-2/res_f_0.032_u085/0.032_u085_8500.txt
18
./pt-brain-2/res_f_0.032_u085/0.032_u085_9000.txt
19
./pt

In [5]:
class PointMPLayer(MessagePassing):
    def __init__(self, in_channels, out_channels):
        # Message passing with "max" aggregation.
        super().__init__(aggr='max')

        # Initialization of the MLP:
        # Here, the number of input features correspond to the hidden node
        # dimensionality plus point dimensionality (=3).
        self.mlp = Sequential(Linear(in_channels + 3, out_channels),
                              ReLU(),
                              Linear(out_channels, out_channels))

    def forward(self, h, pos, edge_index):
        # Start propagating messages.
        return self.propagate(edge_index, h=h, pos=pos)

    def message(self, h_j, pos_j, pos_i):
        # h_j defines the features of neighboring nodes as shape [num_edges, in_channels]
        # pos_j defines the position of neighboring nodes as shape [num_edges, 3]
        # pos_i defines the position of central nodes as shape [num_edges, 3]

        input = pos_j - pos_i  # Compute spatial relation.

        if h_j is not None:
            # In the first layer, we may not have any hidden node features,
            # so we only combine them in case they are present.
            input = torch.cat([h_j, input], dim=-1)

        return self.mlp(input)  # Apply our final MLP.

In [6]:
class Encoder(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = PointMPLayer(3, 64)
        self.conv2 = PointMPLayer(64, 128)
        self.conv3 = PointMPLayer(128, 256)
        self.linear1 = Linear(128, 1)
        self.linear2 = Linear(256, 512)
        self.linear3 = Linear(256, 1)
        self.linear4 = Linear(256, 128)

    def forward(self, pos, edge_index, batch):
        # Compute the kNN graph:
        # Here, we need to pass the batch vector to the function call in order
        # to prevent creating edges between points of different examples.
        # We also add `loop=True` which will add self-loops to the graph in
        # order to preserve central point information.
        #edge_index = knn_graph(pos, k=16, batch=batch, loop=True)

        # Start bipartite message passing.
        h = self.conv1(h=pos, pos=pos, edge_index=edge_index)
        h = h.relu()
        h = self.conv2(h=h, pos=pos, edge_index=edge_index)
        h = h.relu()
        h = self.conv3(h=h, pos=pos, edge_index=edge_index)
        h = h.relu()

        # Global Pooling.
        h_global = global_mean_pool(self.linear3(h), batch)  # [num_examples, hidden_channels]
        h_local = self.linear1(self.linear4(h).relu())
        h_latent = self.linear2(h)
        z = global_mean_pool(h_latent, batch)
        
        return h_global, h_local, h_latent, z, h

In [7]:
class Decoder(torch.nn.Module):
    def __init__(self):
        super().__init__()

        self.conv3 = PointMPLayer(3, 32)
        self.conv4 = PointMPLayer(32, 3)
        self.linear3 = Linear(1, 8759)
        self.linear4 = Linear(1, 8759)
        self.linear5 = Linear(256, 128)
        self.linear6 = Linear(128, 32)
        self.linear7 = Linear(32, 3)
        self.linear8 = Linear(512, 256)

    def forward(self, z):

        #z_topo = self.linear3(z).relu()
        #adj = torch.sigmoid(torch.matmul(z_topo, z_topo.t()))
        #edge_index_d = adj.nonzero().t().contiguous()
        #z_pos = self.linear4(z).relu().reshape(-1,1)
        #z_pos = self.linear5(z_pos)
        z_pos = self.linear5(self.linear8(z).relu()).relu()
        z_pos = self.linear6(z_pos).relu()
        z_pos = self.linear7(z_pos)
        ###edge_index_d = knn_graph(z_pos, k=3, batch=batch, loop=True)
        
        # Start bipartite message passing.
        #h_d = self.conv3(h=z_pos, pos=z_pos, edge_index=edge_index_d)
        #h_d = h_d.relu()
        #h_d = self.conv4(h=h_d, pos=z_pos, edge_index=edge_index_d)
    
        return z_pos

In [8]:
class AE(torch.nn.Module):
    
    def __init__(self, encoder: torch.nn.Module, decoder: torch.nn.Module):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        AE.reset_parameters(self)

    def reset_parameters(self):
        r"""Resets all learnable parameters of the module."""
        reset(self.encoder)
        reset(self.decoder)


    def forward(self, *args, **kwargs) -> Tensor:  # pragma: no cover
        r"""Alias for :meth:`encode`."""
        return self.encoder(*args, **kwargs)
    def encode(self, *args, **kwargs) -> Tensor:
        r"""Runs the encoder and computes node-wise latent variables."""
        return self.encoder(*args, **kwargs)


    def decode(self, *args, **kwargs) -> Tensor:
        r"""Runs the decoder and computes edge probabilities."""
        return self.decoder(*args, **kwargs)

In [9]:
torch.manual_seed(2458)
np.random.seed(2458)
index=np.arange(len(data)).tolist()
np.random.shuffle(index)

In [10]:
index = np.array(index)

In [11]:
train_id = index[:2160]
val_id = index[2160:2440]
train_dataset = [data[i] for i in train_id]
val_dataset = [data[i] for i in val_id]

In [12]:
from torch_geometric.loader import DataLoader

train_loader = DataLoader(train_dataset, batch_size=20, shuffle=True)
test_loader = DataLoader(val_dataset, batch_size=20)

model = AE(Encoder(),Decoder()).to('cuda')
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
mse = torch.nn.MSELoss()  # Define loss criterion.

def train(model, optimizer, loader):
    model.train()

    total_loss = 0
    for data in loader:
        optimizer.zero_grad()  # Clear gradients.
        h_global, h_local, h_latent, z, h = model.encode(data.x, data.edge_index, data.batch)  # Forward pass.
        h_d = model.decode(h_latent)
        loss.backward()  # Backward pass.
        optimizer.step()  # Update model parameters.
        total_loss += loss.item()

    return total_loss / len(train_loader.dataset)


@torch.no_grad()
def test(model, loader):
    model.eval()

    total_loss = 0
    for data in loader:
        h_global, h_local, h_latent, z, h = model.encode(data.x, data.edge_index, data.batch)  # Forward pass.
        h_d = model.decode(h_latent)
        loss = 100*mse(h_global, data.g.reshape(-1,1)) + mse(h_local, data.curv.reshape(-1,1)) + 1000*chamfer_distance(h_d.reshape(-1,8759,3), data.x.reshape(-1,8759,3))[0] # Loss computation.
        total_loss += loss.item()
    return total_loss / len(loader.dataset)

for epoch in range(1, 1001):
    loss = train(model, optimizer, train_loader)
    test_acc = test(model, test_loader)
    print(f'Epoch: {epoch:02d}, Loss: {loss:.4f}, Test Accuracy: {test_acc:.4f}')

/data/home/zhaoyj/DP/dp-1.2.0-gpu/envs/pt-b-gpu3/lib/python3.9/site-packages/torch_geometric/utils/scatter.py:93: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(f"The usage of `scatter(reduce='{reduce}')` "


Epoch: 01, Loss: 83.0751, Test Accuracy: 20.5630
Epoch: 02, Loss: 9.1827, Test Accuracy: 3.9397
Epoch: 03, Loss: 2.9267, Test Accuracy: 3.5408
Epoch: 04, Loss: 2.7723, Test Accuracy: 3.3970
Epoch: 05, Loss: 2.7511, Test Accuracy: 3.3632
Epoch: 06, Loss: 2.6713, Test Accuracy: 3.3105
Epoch: 07, Loss: 2.6288, Test Accuracy: 3.3303
Epoch: 08, Loss: 2.5608, Test Accuracy: 3.1513
Epoch: 09, Loss: 2.4980, Test Accuracy: 3.3526
Epoch: 10, Loss: 2.4717, Test Accuracy: 3.0729
Epoch: 11, Loss: 2.4670, Test Accuracy: 3.4732
Epoch: 12, Loss: 2.4091, Test Accuracy: 3.0573
Epoch: 13, Loss: 2.3927, Test Accuracy: 3.0008
Epoch: 14, Loss: 2.3424, Test Accuracy: 3.0994
Epoch: 15, Loss: 2.3628, Test Accuracy: 2.9361
Epoch: 16, Loss: 2.3182, Test Accuracy: 2.9373
Epoch: 17, Loss: 2.2966, Test Accuracy: 3.0758
Epoch: 18, Loss: 2.2797, Test Accuracy: 2.9583
Epoch: 19, Loss: 2.2731, Test Accuracy: 2.8089
Epoch: 20, Loss: 2.1923, Test Accuracy: 2.7743
Epoch: 21, Loss: 2.1392, Test Accuracy: 2.7433
Epoch: 22, 

In [13]:
torch.save(model.state_dict(), 'SL.pth')